In [1]:
import pandas as pd
from constants import DATA_PATH, EOS_FILE, SENTINEL_FILE

sentinel = pd.read_csv(DATA_PATH / SENTINEL_FILE)
eos = pd.read_csv(DATA_PATH / EOS_FILE)

In [2]:
from constants import X_cols_eos, X_cols_sentinel, y_col

X_sentinel = sentinel[X_cols_sentinel].values
X_eos = eos[X_cols_eos].values

y_sentinel = sentinel[y_col].values
y_eos = eos[y_col].values

In [3]:
import tensorflow as tf

2025-11-02 15:41:49.147854: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [4]:
models = {
    "16, 1": tf.keras.Sequential([
    # Input layer
    tf.keras.Input(shape=(7, )),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1)
]),
    "8, 1": tf.keras.Sequential([
    # Input layer
    tf.keras.Input(shape=(7, )),
    tf.keras.layers.Dense(8, activation='relu'),
    tf.keras.layers.Dense(1)
]),
    "2, 1": tf.keras.Sequential([
    # Input layer
    tf.keras.Input(shape=(7, )),
    tf.keras.layers.Dense(2, activation='relu'),
    tf.keras.layers.Dense(1)
]),
    "4, 1": tf.keras.Sequential([
    # Input layer
    tf.keras.Input(shape=(7, )),
    tf.keras.layers.Dense(4, activation='relu'),
    tf.keras.layers.Dense(1)
]),
    "16, Dropout, 8, Dropout": tf.keras.Sequential([
    # Input layer
    tf.keras.Input(shape=(7, )),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dropout(0.09),
    tf.keras.layers.Dense(8, activation='relu'),
    tf.keras.layers.Dropout(0.09),
    tf.keras.layers.Dense(1)
]),
    "16, Dropout": tf.keras.Sequential([
    # Input layer
    tf.keras.Input(shape=(7, )),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dropout(0.1),
    tf.keras.layers.Dense(1)
])
}

In [ ]:
from model_experiments import PredictionIntervalEstimation

tf.keras.backend.clear_session()

eos_results = {}

for param_string, model in models.items():
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

    exp = PredictionIntervalEstimation(X_eos, y_eos, satellite="EOS-04")
    results = exp.run_experiment(model, model_param_string=param_string, optimizer=optimizer, epochs=1000)
    eos_results[param_string] = results

In [7]:
from constants import OUTPUT_PATH
import json

with open(OUTPUT_PATH / "pi_estimation_censored" / "EOS-04_metrics.json", "w") as f:
    json.dump(eos_results, f, indent=4)

In [ ]:
from model_experiments import PredictionIntervalEstimation
sentinel_results = {}

for param_string, model in models.items():
    tf.keras.backend.clear_session()
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

    exp = PredictionIntervalEstimation(X_sentinel, y_sentinel, satellite="Sentinel-1")
    results = exp.run_experiment(model, model_param_string=param_string, optimizer=optimizer, epochs=1000)
    sentinel_results[param_string] = results

In [9]:
from constants import OUTPUT_PATH
import json

with open(OUTPUT_PATH / "pi_estimation_censored" / "Sentinel-1_metrics.json", "w") as f:
    json.dump(sentinel_results, f, indent=4)